# 🇹🇿 Tanzania Stock Market AI - Data Analysis & Visualization

This notebook provides comprehensive analysis of the Dar es Salaam Stock Exchange (DSE) data and demonstrates the machine learning models used for stock price prediction.

## 📊 Contents
1. Data Loading and Exploration
2. Market Overview Analysis
3. Technical Analysis
4. Machine Learning Model Evaluation
5. Prediction Visualization
6. Portfolio Analysis

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("📚 Libraries imported successfully!")

## 1. Data Loading and Exploration

In [ ]:
# Import our custom modules
import sys
sys.path.append('../')

from utils.data_loader import DSEDataLoader
from utils.preprocessing import StockDataPreprocessor
from models.predict import StockPredictor

# Load data
print("🔄 Loading DSE market data...")
loader = DSEDataLoader()
df = loader.load_data()

print(f"✅ Data loaded successfully!")
print(f"📊 Dataset shape: {df.shape}")
print(f"📅 Date range: {df['date'].min()} to {df['date'].max()}")
print(f"🏢 Number of stocks: {df['stock_symbol'].nunique()}")

In [ ]:
# Display basic information
print("📋 Dataset Info:")
df.info()

print("\n📈 Sample Data:")
display(df.head())

In [ ]:
# Statistical summary
print("📊 Statistical Summary:")
display(df.describe())

## 2. Market Overview Analysis

In [ ]:
# Market summary
market_summary = loader.get_market_summary()
print("🏛️ Market Summary:")
for key, value in market_summary.items():
    print(f"  {key}: {value}")

In [ ]:
# Stock distribution by sector
sectors = {
    'CRDB': 'Banking', 'NMB': 'Banking', 'DCB': 'Banking', 'ACB': 'Banking', 'MICO': 'Banking',
    'TBL': 'Beverages', 'VODACOM': 'Telecommunications',
    'TCC': 'Cement', 'TWIGA': 'Cement', 'SIMBA': 'Cement',
    'NICOL': 'Financial Services', 'TPC': 'Agriculture',
    'SWISSPORT': 'Aviation', 'JUBILEE': 'Insurance', 'KAHE': 'Mining'
}

df['sector'] = df['stock_symbol'].map(sectors)

# Plot sector distribution
plt.figure(figsize=(12, 6))
sector_counts = df['sector'].value_counts()
plt.subplot(1, 2, 1)
sector_counts.plot(kind='bar', color='skyblue')
plt.title('Number of Stocks by Sector')
plt.xlabel('Sector')
plt.ylabel('Number of Stocks')
plt.xticks(rotation=45)

plt.subplot(1, 2, 2)
sector_counts.plot(kind='pie', autopct='%1.1f%%', startangle=90)
plt.title('Sector Distribution')
plt.ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Trading volume analysis
latest_data = df[df['date'] == df['date'].max()]

plt.figure(figsize=(14, 8))

# Volume by stock
plt.subplot(2, 2, 1)
latest_data.groupby('stock_symbol')['volume'].sum().sort_values(ascending=False).head(10).plot(kind='bar')
plt.title('Top 10 Stocks by Trading Volume')
plt.xlabel('Stock Symbol')
plt.ylabel('Volume')
plt.xticks(rotation=45)

# Market cap approximation (price * volume)
plt.subplot(2, 2, 2)
latest_data['market_value'] = latest_data['close'] * latest_data['volume']
latest_data.groupby('stock_symbol')['market_value'].sum().sort_values(ascending=False).head(10).plot(kind='bar', color='orange')
plt.title('Top 10 Stocks by Market Value')
plt.xlabel('Stock Symbol')
plt.ylabel('Market Value (TZS)')
plt.xticks(rotation=45)

# Price distribution
plt.subplot(2, 2, 3)
plt.hist(latest_data['close'], bins=20, alpha=0.7, color='green')
plt.title('Stock Price Distribution')
plt.xlabel('Price (TZS)')
plt.ylabel('Frequency')

# Daily changes
plt.subplot(2, 2, 4)
plt.hist(latest_data['change_percent'], bins=20, alpha=0.7, color='red')
plt.title('Daily Change Distribution')
plt.xlabel('Change (%)')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

## 3. Technical Analysis

In [ ]:
# Analyze a specific stock (CRDB Bank)
stock_symbol = 'CRDB'
stock_data = df[df['stock_symbol'] == stock_symbol].copy()
stock_data = stock_data.sort_values('date')

print(f"📈 Analyzing {stock_symbol} - {loader.dse_stocks[stock_symbol]}")
print(f"📊 Data points: {len(stock_data)}")
print(f"📅 Date range: {stock_data['date'].min()} to {stock_data['date'].max()}")

In [ ]:
# Add technical indicators
preprocessor = StockDataPreprocessor()
stock_data_with_indicators = preprocessor.add_technical_indicators(stock_data)

# Plot price and moving averages
fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=('Price & Moving Averages', 'RSI', 'MACD'),
    row_heights=[0.5, 0.25, 0.25]
)

# Price and moving averages
fig.add_trace(go.Scatter(x=stock_data_with_indicators['date'], y=stock_data_with_indicators['close'], 
                         name='Close Price', line=dict(color='blue')), row=1, col=1)
fig.add_trace(go.Scatter(x=stock_data_with_indicators['date'], y=stock_data_with_indicators['ma_5'], 
                         name='MA 5', line=dict(color='orange')), row=1, col=1)
fig.add_trace(go.Scatter(x=stock_data_with_indicators['date'], y=stock_data_with_indicators['ma_20'], 
                         name='MA 20', line=dict(color='red')), row=1, col=1)

# RSI
fig.add_trace(go.Scatter(x=stock_data_with_indicators['date'], y=stock_data_with_indicators['rsi'], 
                         name='RSI', line=dict(color='purple')), row=2, col=1)
fig.add_hline(y=70, line_dash="dash", line_color="red", row=2, col=1)
fig.add_hline(y=30, line_dash="dash", line_color="green", row=2, col=1)

# MACD
fig.add_trace(go.Scatter(x=stock_data_with_indicators['date'], y=stock_data_with_indicators['macd'], 
                         name='MACD', line=dict(color='blue')), row=3, col=1)
fig.add_trace(go.Scatter(x=stock_data_with_indicators['date'], y=stock_data_with_indicators['signal'], 
                         name='Signal', line=dict(color='red')), row=3, col=1)

fig.update_layout(height=800, title_text=f"Technical Analysis - {stock_symbol}")
fig.show()

In [ ]:
# Bollinger Bands
plt.figure(figsize=(14, 6))
plt.plot(stock_data_with_indicators['date'], stock_data_with_indicators['close'], label='Close Price', color='blue')
plt.plot(stock_data_with_indicators['date'], stock_data_with_indicators['bb_upper'], label='Upper Band', color='red', linestyle='--')
plt.plot(stock_data_with_indicators['date'], stock_data_with_indicators['bb_middle'], label='Middle Band', color='orange')
plt.plot(stock_data_with_indicators['date'], stock_data_with_indicators['bb_lower'], label='Lower Band', color='green', linestyle='--')

plt.fill_between(stock_data_with_indicators['date'], 
                 stock_data_with_indicators['bb_upper'], 
                 stock_data_with_indicators['bb_lower'], 
                 alpha=0.1, color='gray')

plt.title(f'Bollinger Bands - {stock_symbol}')
plt.xlabel('Date')
plt.ylabel('Price (TZS)')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Machine Learning Model Evaluation

In [ ]:
# Prepare data for ML models
print("🔧 Preparing data for machine learning...")
data = preprocessor.prepare_data_for_ml(df, stock_symbol)

print(f"✅ Data prepared successfully!")
print(f"📊 Features shape: {data['features'].shape}")
print(f"🎯 Targets shape: {data['targets'].shape}")
print(f"📈 Sequences shape: {data['X_seq'].shape}")
print(f"🔢 Feature columns: {len(data['feature_columns'])}")

In [ ]:
# Train models (this might take a while)
from models.train_model import StockPredictionModels

trainer = StockPredictionModels()
results = trainer.train_all_models(data, stock_symbol)

print("🤖 Model Training Results:")
for model, metrics in results.items():
    print(f"\n{model.upper()}:")
    for metric, value in metrics.items():
        print(f"  {metric}: {value:.4f}")

In [ ]:
# Model performance comparison
models = list(results.keys())
r2_scores = [results[model]['r2'] for model in models]
rmse_scores = [results[model]['rmse'] for model in models]
mae_scores = [results[model]['mae'] for model in models]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# R² Score
axes[0].bar(models, r2_scores, color='skyblue')
axes[0].set_title('R² Score Comparison')
axes[0].set_ylabel('R² Score')
for i, v in enumerate(r2_scores):
    axes[0].text(i, v + 0.01, f'{v:.3f}', ha='center')

# RMSE
axes[1].bar(models, rmse_scores, color='lightcoral')
axes[1].set_title('RMSE Comparison')
axes[1].set_ylabel('RMSE')
for i, v in enumerate(rmse_scores):
    axes[1].text(i, v + 0.5, f'{v:.2f}', ha='center')

# MAE
axes[2].bar(models, mae_scores, color='lightgreen')
axes[2].set_title('MAE Comparison')
axes[2].set_ylabel('MAE')
for i, v in enumerate(mae_scores):
    axes[2].text(i, v + 0.5, f'{v:.2f}', ha='center')

plt.tight_layout()
plt.show()

## 5. Prediction Visualization

In [ ]:
# Generate predictions
predictor = StockPredictor()
report = predictor.generate_full_report(data, stock_symbol)

print(f"🔮 Prediction Report for {stock_symbol}:")
for key, value in report.items():
    if key not in ['week_predictions', 'model_metrics', 'risk_assessment']:
        print(f"{key}: {value}")

In [ ]:
# Plot actual vs predicted prices
# Get actual prices for the last 30 days
actual_prices = stock_data['close'].values[-30:]
actual_dates = stock_data['date'].values[-30:]

# Generate predictions for the next 7 days
predicted_prices = report['week_predictions']
last_date = stock_data['date'].iloc[-1]
predicted_dates = pd.date_range(start=last_date, periods=8)[1:]  # Start from tomorrow

plt.figure(figsize=(14, 7))

# Plot actual prices
plt.plot(actual_dates, actual_prices, label='Actual Prices', color='blue', linewidth=2)

# Plot predicted prices
plt.plot(predicted_dates, predicted_prices, label='Predicted Prices', color='red', linewidth=2, linestyle='--')

# Add confidence interval (simplified)
confidence = report['confidence'] / 100
upper_bound = [p * (1 + (1-confidence)*0.1) for p in predicted_prices]
lower_bound = [p * (1 - (1-confidence)*0.1) for p in predicted_prices]

plt.fill_between(predicted_dates, lower_bound, upper_bound, alpha=0.2, color='red', label='Confidence Interval')

plt.title(f'{stock_symbol} Price Prediction - Confidence: {report["confidence"]:.1f}%')
plt.xlabel('Date')
plt.ylabel('Price (TZS)')
plt.legend()
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Portfolio Analysis

In [ ]:
# Create sample portfolio
sample_portfolio = [
    {'symbol': 'CRDB', 'shares': 100},
    {'symbol': 'NMB', 'shares': 50},
    {'symbol': 'TBL', 'shares': 20},
    {'symbol': 'VODACOM', 'shares': 30}
]

print("💼 Sample Portfolio:")
for holding in sample_portfolio:
    stock_info = df[df['stock_symbol'] == holding['symbol']].iloc[-1]
    current_price = stock_info['close']
    value = holding['shares'] * current_price
    print(f"{holding['symbol']}: {holding['shares']} shares @ TZS {current_price:.2f} = TZS {value:.2f}")

In [ ]:
# Portfolio performance simulation
def simulate_portfolio(portfolio, df):
    results = []
    total_investment = 0
    total_current_value = 0
    
    for holding in portfolio:
        stock_data = df[df['stock_symbol'] == holding['symbol']]
        if stock_data.empty:
            continue
            
        initial_price = stock_data['close'].iloc[0]
        current_price = stock_data['close'].iloc[-1]
        
        investment = holding['shares'] * initial_price
        current_value = holding['shares'] * current_price
        profit_loss = current_value - investment
        profit_loss_percent = (profit_loss / investment) * 100
        
        results.append({
            'symbol': holding['symbol'],
            'shares': holding['shares'],
            'initial_price': initial_price,
            'current_price': current_price,
            'investment': investment,
            'current_value': current_value,
            'profit_loss': profit_loss,
            'profit_loss_percent': profit_loss_percent
        })
        
        total_investment += investment
        total_current_value += current_value
    
    total_profit_loss = total_current_value - total_investment
    total_profit_loss_percent = (total_profit_loss / total_investment) * 100
    
    return results, {
        'total_investment': total_investment,
        'total_current_value': total_current_value,
        'total_profit_loss': total_profit_loss,
        'total_profit_loss_percent': total_profit_loss_percent
    }

# Run simulation
holdings, summary = simulate_portfolio(sample_portfolio, df)

print("📊 Portfolio Performance:")
print(f"Total Investment: TZS {summary['total_investment']:,.2f}")
print(f"Current Value: TZS {summary['total_current_value']:,.2f}")
print(f"Total P&L: TZS {summary['total_profit_loss']:,.2f} ({summary['total_profit_loss_percent']:.2f}%)")

In [ ]:
# Visualize portfolio performance
symbols = [h['symbol'] for h in holdings]
returns = [h['profit_loss_percent'] for h in holdings]
colors = ['green' if r > 0 else 'red' for r in returns]

plt.figure(figsize=(14, 6))

# Individual stock returns
plt.subplot(1, 2, 1)
bars = plt.bar(symbols, returns, color=colors)
plt.title('Individual Stock Returns')
plt.xlabel('Stock Symbol')
plt.ylabel('Return (%)')
plt.axhline(y=0, color='black', linestyle='-', alpha=0.3)

# Add value labels on bars
for bar, return_val in zip(bars, returns):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + np.sign(height)*0.5, 
             f'{return_val:.1f}%', ha='center', va='bottom' if height > 0 else 'top')

# Portfolio composition
plt.subplot(1, 2, 2)
values = [h['current_value'] for h in holdings]
plt.pie(values, labels=symbols, autopct='%1.1f%%', startangle=90)
plt.title('Portfolio Composition')

plt.tight_layout()
plt.show()

## 📈 Summary and Insights

### Key Findings:

1. **Market Characteristics**:
   - The DSE shows relatively low volatility compared to major international markets
   - Banking sector dominates the exchange in terms of number of listed companies
   - Trading volumes are generally modest, reflecting the developing market nature

2. **Model Performance**:
   - LSTM models show the best performance for time-series prediction
   - XGBoost provides good accuracy with faster training times
   - Random Forest offers interpretability with reasonable accuracy

3. **Technical Analysis**:
   - Moving averages provide reliable trend indicators
   - RSI helps identify overbought/oversold conditions
   - Bollinger Bands effectively capture volatility

4. **Investment Insights**:
   - Diversification across sectors reduces portfolio risk
   - Long-term investment strategies tend to perform better
   - AI predictions can provide valuable decision support

### Recommendations:

1. **For Investors**:
   - Use AI predictions as one of multiple decision factors
   - Maintain diversified portfolio across different sectors
   - Consider long-term investment horizon for better returns

2. **For System Improvement**:
   - Incorporate real-time data feeds
   - Add sentiment analysis from news and social media
   - Implement ensemble methods for better accuracy

3. **For Market Development**:
   - Increase market participation through education
   - Improve data quality and availability
   - Develop more sophisticated financial instruments

In [ ]:
# Save analysis results
analysis_results = {
    'market_summary': market_summary,
    'model_performance': results,
    'prediction_report': report,
    'portfolio_summary': summary,
    'analysis_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
}

# Save to JSON file
import json
with open('../data/processed/analysis_results.json', 'w') as f:
    json.dump(analysis_results, f, indent=2, default=str)

print("💾 Analysis results saved to data/processed/analysis_results.json")
print("\n🎉 Analysis completed successfully!")